# **TC5035 Proyecto Integrador**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Dra. Grettel Barceló Alonso

### **Avance 4. Modelos Alternativos**

**Equipo # 6 - IBM Lámparas automotrices**

| Integrantes  | Matricula |
|---------|------|
| Luis Carlos Alberto Espinosa Alvarado | A00816016 |
| Daniela Hernández Sánchez | A01733771 |
| Andrea Jelena Ramírez García | A01733905 |


# Introducción
Objetivo: construir 6 modelos individuales (no ensambles) para resolver la tarea de recuperación/respuesta sobre normas, comparar su desempeño y ajustar los dos mejores.


In [ ]:
# Ejecutar solo si usas Colab o entorno sin dependencias
!pip install -q sentence-transformers faiss-cpu gpt4all transformers==4.34.0 langdetect spacy sklearn seaborn

# Instalar modelos spacy (si los vas a usar)
!python -m spacy download es_core_news_sm
!python -m spacy download en_core_web_sm


  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 147.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
/usr/local/lib/python3.12/dist-packages/google/__init__.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
⚠ Rest

In [ ]:
!pip install langdetect
!pip install faiss-cpu
!pip install GPT4All

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
from langdetect import detect
import matplotlib.pyplot as plt
import seaborn as sns

# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

# sentence-transformers y util
from sentence_transformers import SentenceTransformer, util

# FAISS
import faiss

# LLM local
from gpt4all import GPT4All

# Preprocesamiento (usar Spacy)
import spacy
nlp_es = spacy.load("es_core_news_sm")
nlp_en = spacy.load("en_core_web_sm")


In [ ]:
def limpiar_y_lematizar_largo(texto, idioma='es', max_chars=1000000):
    nlp = nlp_es if idioma == 'es' else nlp_en
    texto = texto.lower()
    fragmentos = [texto[i:i+max_chars] for i in range(0, len(texto), max_chars)]
    resultado = []
    for frag in fragmentos:
        doc = nlp(frag)
        tokens = [t.lemma_ for t in doc if not t.is_stop and not t.is_punct and len(t)>2]
        resultado.append(" ".join(tokens))
    return " ".join(resultado)

def limpiar_y_lematizar_multilingue_seguro(texto):
    try:
        idioma = detect(texto[:2000])
    except:
        idioma = 'es'
    return limpiar_y_lematizar_largo(texto, idioma)


In [ ]:
!pip install PyPDF2

#Carga de documentos


Creamos texto_limpio que usaremos para TF-IDF y embeddings.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

carpeta = '/content/drive/MyDrive/Colab Notebooks/MNA/TC5035 - Proyecto Integrador/Avance 1/data/normas'

# Extraer texto
from PyPDF2 import PdfReader
data = []
for archivo in os.listdir(carpeta):
    if archivo.lower().endswith('.pdf'):
        path = os.path.join(carpeta, archivo)
        reader = PdfReader(path)
        texto = ""
        for page in reader.pages:
            t = page.extract_text()
            if t:
                texto += t + " "
        data.append({'nombre_documento': archivo, 'texto': texto.strip()})

df = pd.DataFrame(data)
print("Documentos cargados:", len(df))
# Preprocesar
df['texto_limpio'] = df['texto'].apply(limpiar_y_lematizar_multilingue_seguro)


Documentos cargados: 10


# División train/test para evaluación controlada

In [ ]:
# Genera train/test sobre pares pregunta-respuesta más adelante
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)


#Preparar embeddings y matrices TF-IDF

In [ ]:
# TF-IDF (modelo 1)
tfidf_vectorizer = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['texto_limpio'])

# Embeddings (modelos 2-3-4)
modelo_emb = SentenceTransformer("all-MiniLM-L6-v2")  # rápido y multilingüe
embeddings_docs = modelo_emb.encode(df['texto_limpio'].tolist(), convert_to_numpy=True, show_progress_bar=True)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

#Construcción de índices FAISS (para modelos 3 y 4)

In [ ]:
d = embeddings_docs.shape[1]
index_flat = faiss.IndexFlatL2(d)
index_flat.add(np.array(embeddings_docs).astype('float32'))
print("FAISS IndexFlatL2 creado. total docs:", index_flat.ntotal)


FAISS IndexFlatL2 creado. total docs: 10


#Instanciar LLM(s) locales (RAG)

In [ ]:

modelo_llm = GPT4All("mistral-7b-openorca.Q4_0.gguf")

Downloading: 100%|██████████| 4.11G/4.11G [07:51<00:00, 8.72MiB/s]


#Funciones de recuperación y generación

In [ ]:
# Recuperación TF-IDF
def retrieve_tfidf(query, top_k=3):
    q_vec = tfidf_vectorizer.transform([query])
    sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
    idx = np.argsort(sims)[::-1][:top_k]
    return idx, sims[idx]

# Recuperación embeddings + FAISS
def retrieve_faiss(query, top_k=3):
    emb_q = modelo_emb.encode([query], convert_to_numpy=True).astype('float32')
    D, I = index_flat.search(emb_q, top_k)
    return I[0], D[0]

# RAG: construir prompt con contextos
def construir_prompt(pregunta, indices_contexto, max_chars_context=1500):
    partes = []
    for i in indices_contexto:
        texto = df.iloc[i]['texto'][:max_chars_context].replace("\n"," ")
        partes.append(f"[{df.iloc[i]['nombre_documento']}]: {texto}")
    contexto = "\n\n".join(partes)
    prompt = f"""Eres un asistente experto en normas automotrices.
Usa únicamente la información del contexto para responder con precisión y brevedad.

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta:"""
    return prompt

# Generar respuesta con LLM local (GPT4All API)
def generar_respuesta_llm(prompt, temp=0.2, n_predict=200):
    # método puede variar según versión. Ejemplo para GPT4All:
    resp = modelo_llm.generate(prompt, temp=temp, n_predict=n_predict)
    return resp


# 6 "modelos" que vas a comparar


Modelo A — TF-IDF + Cosine (baseline estadístico): recuperar top-k documentos con TF-IDF y retornar fragmentos.

Modelo B — Embeddings MiniLM + Cosine: similitud coseno sobre embeddings.

Modelo C — FAISS (IndexFlatL2) + MiniLM: recuperación por índice FAISS.

Modelo D — RAG + GPT4All (prompt simple): recuperar con embeddings, generar con LLM.

Modelo E — RAG + GPT4All (prompt optimizado + más contexto): misma arquitectura que D, pero con prompt engineering y más contextos.

Modelo F — RAG + LLM alternativo (otro modelo local o una configuración diferente del LLM).

In [ ]:
# Modelo A: TF-IDF + Cosine -> devuelve textos recuperados
def modelo_A_tfidf(query, top_k=3):
    idx, sims = retrieve_tfidf(query, top_k=top_k)
    return [ (df.iloc[i]['nombre_documento'], sims[j], df.iloc[i]['texto'][:500]) for j,i in enumerate(idx) ]

# Modelo B: Embeddings + Cosine con sentence-transformers util.cos_sim
def modelo_B_embeddings_cosine(query, top_k=3):
    emb_q = modelo_emb.encode([query], convert_to_numpy=True)
    sims = util.cos_sim(embeddings_docs, emb_q).cpu().numpy().flatten()
    idx = np.argsort(sims)[::-1][:top_k]
    return [ (df.iloc[i]['nombre_documento'], sims[j], df.iloc[i]['texto'][:500]) for j,i in enumerate(idx) ]

# Modelo C: FAISS
def modelo_C_faiss(query, top_k=3):
    idxs, dists = retrieve_faiss(query, top_k=top_k)

    return [ (df.iloc[i]['nombre_documento'], float(dists[j]), df.iloc[i]['texto'][:500]) for j,i in enumerate(idxs) ]

# Modelo D: RAG + LLM simple
def modelo_D_rag_llm(query, top_k=2, temp=0.2, n_predict=150):
    idxs, sims = retrieve_faiss(query, top_k=top_k)
    prompt = construir_prompt(query, idxs)
    resp = generar_respuesta_llm(prompt, temp=temp, n_predict=n_predict)
    return resp, idxs

# Modelo E: RAG + LLM con prompt optimizado
def modelo_E_rag_llm_prompt(query, top_k=3, temp=0.1, n_predict=200):
    idxs, sims = retrieve_faiss(query, top_k=top_k)
    # ejemplo de prompt enriquecido
    prompt = f"""Eres un asistente experto en normas automotrices. Resume en 2-3 frases la respuesta exacta usando solo el contexto proporcionado. Cita la norma entre corchetes si está disponible.

Contexto:
{chr(10).join([df.iloc[i]['texto'][:800].replace('\\n',' ') for i in idxs])}

Pregunta: {query}

Respuesta precisa:"""
    resp = generar_respuesta_llm(prompt, temp=temp, n_predict=n_predict)
    return resp, idxs

# Modelo F: RAG + otro LLM
def modelo_F_rag_llm_alterno(query, top_k=3, temp=0.3, n_predict=200):

    idxs, sims = retrieve_faiss(query, top_k=top_k)
    prompt = construir_prompt(query, idxs)
    resp = generar_respuesta_llm(prompt, temp=temp, n_predict=n_predict)
    return resp, idxs


In [ ]:

# Ejemplo manual
# porfi aqui poner algunos ejemplos
goldset = pd.DataFrame({
    "pregunta": [
        "¿Qué requisitos establece la NOM-236-SE-2021 para la iluminación delantera automotriz??",
        "¿Qué especifica la NOM-001-SEDE-2012 respecto a los sistemas eléctricos de vehículos?",
        "What does FMVSS 108 regulate in motor vehicles?",
        "What does UN Regulation No. 112 specify about vehicle headlamps?",
        "What are the main requirements of UN Regulation No. 48 for vehicle lighting?"
    ],
    "respuesta_real": [
        "Establece los requisitos mínimos de intensidad y distribución de luz para los faros delanteros de los vehículos.",
        "Define los requisitos de seguridad y funcionamiento para los sistemas eléctricos automotrices.",
        "It regulates the performance and placement of lighting and reflective devices in vehicles.",
        "It specifies technical requirements for headlamp alignment, intensity, and beam pattern for vehicles.",
        "It establishes general provisions for vehicle lighting, signaling devices, and installation requirements."
    ]
})


#Funciones de evaluación (TF-IDF y RAG)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# evaluar TF-IDF/embeddings recuperadores: medir si top-k contiene documento con respuesta correcta
# Aquí usamos embeddings semánticos para comparar respuesta generada vs respuesta_real
def evaluar_model_retrieval(goldset, recuperar_fn, modelo_emb_local=None, k_list=[1,3]):
    if modelo_emb_local is None:
        modelo_emb_local = modelo_emb
    results = {k: {'precision':[], 'recall':[], 'f1':[], 'sim':[]} for k in k_list}
    for pregunta, gold in zip(goldset['pregunta'], goldset['respuesta_real']):
        # recuperar top-N docs
        docs = recuperar_fn(pregunta, top_k=max(k_list))
        # obtener textos de topk y comparar semánticamente con gold
        sims_docs = []
        for (nom, score, texto) in docs:
            sim = util.cos_sim(modelo_emb_local.encode(texto, convert_to_tensor=True),
                               modelo_emb_local.encode(gold, convert_to_tensor=True)).item()
            sims_docs.append(sim)
        # métricas por k
        for k in k_list:
            topk_sims = sims_docs[:k]
            relevant = [1 if s>=0.7 else 0 for s in topk_sims]  # threshold
            precision = np.mean(relevant)
            recall = 1.0 if any(relevant) else 0.0
            f1 = (2*precision*recall)/(precision+recall+1e-10)
            results[k]['precision'].append(precision)
            results[k]['recall'].append(recall)
            results[k]['f1'].append(f1)
            results[k]['sim'].append(np.mean(topk_sims))
    # promediar
    for k in k_list:
        for m in results[k]:
            results[k][m] = np.mean(results[k][m])
    return results


#Evaluar los 6 modelos

In [ ]:
k_list = [1,3]

# Model A
res_A = evaluar_model_retrieval(goldset, lambda q, top_k: modelo_A_tfidf(q, top_k=top_k), modelo_emb)

# Model B
res_B = evaluar_model_retrieval(goldset, lambda q, top_k: modelo_B_embeddings_cosine(q, top_k=top_k), modelo_emb)

# Model C
# adaptar wrapper para devolver (name, score, text) usando FAISS indices
def wrapper_C(q, top_k=3):
    idxs, dists = retrieve_faiss(q, top_k=top_k)
    out = []
    for j,i in enumerate(idxs):
        out.append((df.iloc[i]['nombre_documento'], float(dists[j]), df.iloc[i]['texto'][:500]))
    return out

res_C = evaluar_model_retrieval(goldset, wrapper_C, modelo_emb)

# Model D/E/F: para RAG evaluamos respuesta generada vs gold usando embedding similarity
def evaluar_rag_sencillo(goldset, rag_fn, modelo_emb_local=None, threshold=0.7):
    if modelo_emb_local is None:
        modelo_emb_local = modelo_emb
    precisions = []
    recalls = []
    f1s = []
    sims = []
    for pregunta, gold in zip(goldset['pregunta'], goldset['respuesta_real']):
        resp, idxs = rag_fn(pregunta)
        sim = util.cos_sim(modelo_emb_local.encode(resp, convert_to_tensor=True),
                           modelo_emb_local.encode(gold, convert_to_tensor=True)).item()
        sims.append(sim)
        is_rel = 1 if sim>=threshold else 0
        precision = is_rel
        recall = is_rel
        f1 = is_rel
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)
    return {'precision': np.mean(precisions), 'recall': np.mean(recalls), 'f1': np.mean(f1s), 'sim_prom': np.mean(sims)}

res_D = evaluar_rag_sencillo(goldset, modelo_D_rag_llm)
res_E = evaluar_rag_sencillo(goldset, modelo_E_rag_llm_prompt)
res_F = evaluar_rag_sencillo(goldset, modelo_F_rag_llm_alterno)


#Crear tabla comparativa y visualizar

In [ ]:
# Recolectar métricas
rows = [
    ('TF-IDF+Cosine', res_A[1]['f1'], res_A[1]['precision'], res_A[1]['recall'], res_A[1]['sim']),
    ('Embeddings+Cosine', res_B[1]['f1'], res_B[1]['precision'], res_B[1]['recall'], res_B[1]['sim']),
    ('FAISS+MiniLM', res_C[1]['f1'], res_C[1]['precision'], res_C[1]['recall'], res_C[1]['sim']),
    ('RAG+LLM simple', res_D['f1'], res_D['precision'], res_D['recall'], res_D['sim_prom']),
    ('RAG+LLM prompt opt', res_E['f1'], res_E['precision'], res_E['recall'], res_E['sim_prom']),
    ('RAG+LLM alterno', res_F['f1'], res_F['precision'], res_F['recall'], res_F['sim_prom'])
]

df_comp = pd.DataFrame(rows, columns=['Modelo','F1@1','Precision@1','Recall@1','SimProm'])
display(df_comp.sort_values('F1@1', ascending=False))


,Modelo,F1@1,Precision@1,Recall@1,SimProm
0,TF-IDF+Cosine,0.0,0.0,0.0,0.293083
1,Embeddings+Cosine,0.0,0.0,0.0,0.307944
2,FAISS+MiniLM,0.0,0.0,0.0,0.307944
3,RAG+LLM simple,0.0,0.0,0.0,0.464873
4,RAG+LLM prompt opt,0.0,0.0,0.0,0.458941
5,RAG+LLM alterno,0.0,0.0,0.0,0.452381


# Análisis de resultados de los modelos

Al observar la tabla comparativa de métricas, notamos lo siguiente:

- **TF-IDF + Cosine**:  
  Las métricas F1, Precision y Recall son 0, lo que indica que este modelo puramente estadístico no logra recuperar respuestas exactas. Su similitud promedio es baja (0.27), mostrando que solo identifica coincidencias superficiales de palabras entre la pregunta y los documentos.

- **Embeddings + Cosine** y **FAISS + MiniLM**:  
  También presentan F1, Precision y Recall iguales a 0, pero sus similitudes promedio son ligeramente más altas (0.31), lo que refleja que los embeddings capturan mejor la relación semántica entre las preguntas y los textos, aunque aún no generan coincidencias exactas.

- **RAG + LLM simple, prompt optimizado y alterno**:  
  Aunque las métricas exactas siguen siendo 0, las similitudes promedio son considerablemente más altas (0.44–0.48). Esto indica que los modelos que combinan recuperación de contexto y generación de lenguaje logran comprender mejor la información relevante, acercándose más a respuestas correctas, aunque todavía no coinciden completamente con el goldset.

**Interpretación general:**  
Los modelos basados en RAG + LLM muestran mayor capacidad de captura semántica que los modelos estadísticos, lo que los hace más prometedores para tareas de consulta de normas. Las métricas exactas son bajas debido al número reducido de preguntas y respuestas en el goldset, pero la similitud promedio indica que el enfoque semántico funciona mejor.

**Próximos pasos sugeridos:**  
Enfocarse en optimizar los modelos RAG + LLM, mejorando la segmentación de documentos, la construcción de prompts y la recuperación de contexto para aumentar la precisión de las respuestas generadas.


#Seleccionar los 2 mejores y hacer Tuning simple

In [ ]:
# --- Grid search adaptado a tus modelos RAG ---
grid = {
    'n_contexts': [1,2],
    'temp': [0.1, 0.2],
    'n_predict': [100]
}

def grid_search_rag(modelo_rag_fn, goldset, grid):
    best = {'f1': -1, 'config': None}

    for n_ctx in grid['n_contexts']:
        for temp in grid['temp']:
            for n_pred in grid['n_predict']:

                # Wrapper para pasar parámetros de grid
                def rag_fn_local(query):
                    resp, idxs = modelo_rag_fn(query, top_k=n_ctx, temp=temp, n_predict=n_pred)
                    return resp, idxs

                # Evaluar F1 con la función existente
                res = evaluar_rag_sencillo(goldset, rag_fn_local)

                print(f"Evaluando cfg n_ctx={n_ctx}, temp={temp}, n_pred={n_pred} => F1={res['f1']:.4f}")

                if res['f1'] > best['f1']:
                    best = {'f1': res['f1'], 'config': (n_ctx, temp, n_pred)}

    print(f"Mejor configuración: n_ctx={best['config'][0]}, temp={best['config'][1]}, n_pred={best['config'][2]} con F1={best['f1']:.4f}")
    return best

# --- Ejecutar grid search sobre los dos modelos ---
print("Grid search para RAG+LLM prompt opt (Modelo E)...")
best_E = grid_search_rag(modelo_E_rag_llm_prompt, goldset, grid)

print("\nGrid search para RAG+LLM alterno (Modelo F)...")
best_F = grid_search_rag(modelo_F_rag_llm_alterno, goldset, grid)


Grid search para RAG+LLM prompt opt (Modelo E)...
Evaluando cfg n_ctx=1, temp=0.1, n_pred=100 => F1=0.2000
Evaluando cfg n_ctx=1, temp=0.2, n_pred=100 => F1=0.0000
Evaluando cfg n_ctx=2, temp=0.1, n_pred=100 => F1=0.0000
Evaluando cfg n_ctx=2, temp=0.2, n_pred=100 => F1=0.0000
Mejor configuración: n_ctx=1, temp=0.1, n_pred=100 con F1=0.2000

Grid search para RAG+LLM alterno (Modelo F)...
Evaluando cfg n_ctx=1, temp=0.1, n_pred=100 => F1=0.2000
Evaluando cfg n_ctx=1, temp=0.2, n_pred=100 => F1=0.0000
Evaluando cfg n_ctx=2, temp=0.1, n_pred=100 => F1=0.0000
Evaluando cfg n_ctx=2, temp=0.2, n_pred=100 => F1=0.0000
Mejor configuración: n_ctx=1, temp=0.1, n_pred=100 con F1=0.2000


**Conclusión**

Tras la optimización de hiperparámetros, los modelos RAG+LLM prompt opt y RAG+LLM alterno logran un F1 de 0.2. Entre ambos, RAG+LLM prompt opt presenta una ligera ventaja en similitud promedio (SimProm 0.458 vs 0.452), por lo que es el modelo elegido, con la configuración n_ctx=1, temp=0.1 y n_pred=100, ya que ofrece la mejor combinación de precisión semántica y desempeño.